In [ ]:
# --- project bootstrap ---
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(ROOT)
# -------------------------


In [ ]:
# Cell 1: Helpers
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Mapping, Tuple

import matplotlib.pyplot as plt
import numpy as np

import fgtn.classA_U1FGTN as classA_U1FGTN
from fgtn.classA_U1FGTN import classA_U1FGTN
def load_final_time_samples(npz_path: Path) -> np.ndarray:
    """Return final-time G samples with shape (n_samples, Nlayer, Nlayer)."""
    with np.load(npz_path) as data:
        G_hist = data["G_hist"]
    return G_hist[:, -1]


def infer_Nx_Ny_from_filename(npz_path: Path, Nlayer: int) -> Tuple[int, int]:
    stem = npz_path.name
    import re
    m = re.search(r"N(\d+)x(\d+)", stem)
    if m:
        return int(m.group(1)), int(m.group(2))
    # fallback: solve Nlayer = 2 * Nx * Ny
    for cand in range(1, Nlayer + 1):
        if Nlayer % (2 * cand) == 0:
            Ny = Nlayer // (2 * cand)
            return cand, Ny
    raise ValueError(f"Could not factor Nlayer={Nlayer} into 2*Nx*Ny.")


def build_sub_indices(Nx: int, Ny: int, y_cut: int) -> Tuple[np.ndarray, np.ndarray]:
    yA = np.arange(y_cut, Ny)
    sub_indices = []
    for y in yA:
        base = 2 * Nx * y
        for x in range(Nx):
            sub_indices.append(base + 2 * x)
            sub_indices.append(base + 2 * x + 1)
    return np.array(sub_indices, dtype=int), yA


def entanglement_contour_batch(Gtt_batch: np.ndarray, Nx: int, Ny: int) -> np.ndarray:
    """
    Gtt_batch: (B, Nlayer, Nlayer)
    Returns: (B, Nx, Ny)
    """
    arr = np.asarray(Gtt_batch, dtype=np.complex128)
    if arr.ndim != 3:
        raise ValueError(f"expected (B,N,N); got {arr.shape}")

    B, Nlayer, _ = arr.shape
    I = np.eye(Nlayer, dtype=np.complex128)
    G2 = 0.5 * (I + arr)

    evals, vecs = np.linalg.eigh(G2)
    evals = np.clip(np.real_if_close(evals), 1e-12, 1 - 1e-12)
    f_eigs = -(evals * np.log(evals) + (1.0 - evals) * np.log(1.0 - evals))

    # diag(F) = sum_k f_eigs[k] * |vecs[i,k]|^2
    diagF = np.einsum("bik,bk,bik->bi", vecs, f_eigs, vecs.conj(), optimize=True).real

    diagF = diagF.reshape(B, 2, Nx, Ny, order="F")
    return diagF.sum(axis=1)  # (B, Nx, Ny)


def compute_entanglement_contour_samples(
    final_samples: np.ndarray,
    Nx: int,
    Ny: int,
    y_cut: int,
    batch_size: int = 8,
) -> np.ndarray:
    """Return contour samples with shape (n_samples, Nx, Ny_sub)."""
    sub_indices, yA = build_sub_indices(Nx, Ny, y_cut)
    Ny_sub = len(yA)

    n_samples = final_samples.shape[0]
    out = np.zeros((n_samples, Nx, Ny_sub), dtype=float)

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        G_sub_batch = np.stack(
            [final_samples[s][np.ix_(sub_indices, sub_indices)] for s in range(start, end)],
            axis=0,
        )
        s_batch = entanglement_contour_batch(G_sub_batch, Nx, Ny_sub)
        out[start:end] = s_batch

    return out


def frobenius_norm(matrix: np.ndarray) -> float:
    return float(np.linalg.norm(matrix, ord="fro"))


def occupation_trace(matrix: np.ndarray) -> float:
    # works for non-square by taking the min dimension
    return float(np.trace(matrix).real)


def spectral_radius(matrix: np.ndarray) -> float:
    if matrix.shape[0] == matrix.shape[1]:
        eigvals = np.linalg.eigvals(matrix)
        return float(np.max(np.abs(eigvals)))
    # fallback for rectangular: largest singular value
    return float(np.linalg.svd(matrix, compute_uv=False)[0])


def frobenius_standard_error(samples: np.ndarray) -> float:
    """Standard error of Frobenius norm across raw samples."""
    count = samples.shape[0]
    if count <= 1:
        return float("nan")
    sample_norms = np.sqrt(np.sum(np.abs(samples) ** 2, axis=tuple(range(1, samples.ndim))))
    return float(sample_norms.std(ddof=1) / np.sqrt(count))


def mean_estimator(samples: np.ndarray) -> np.ndarray:
    return samples.mean(axis=0)


def median_of_means_estimator(samples: np.ndarray, n_groups: int = None) -> np.ndarray:
    """Elementwise median-of-means estimator."""
    count = samples.shape[0]
    if n_groups is None:
        n_groups = max(1, int(np.sqrt(count)))
    n_groups = max(1, min(n_groups, count))
    groups = [grp for grp in np.array_split(samples, n_groups, axis=0) if grp.size > 0]
    group_means = np.stack([grp.mean(axis=0) for grp in groups], axis=0)
    return np.median(group_means, axis=0)


def analyze_estimator(
    samples: np.ndarray,
    sample_counts: Iterable[int],
    estimator_fn: Callable[[np.ndarray], np.ndarray],
) -> Mapping[str, List[float]]:
    """Apply estimator over increasing sample counts and collect diagnostics."""
    metrics = {
        "frobenius_norm": [],
        "frobenius_se": [],
        "successive_diff": [],
        "occupation_trace": [],
        "spectral_radius": [],
    }
    previous = None

    for count in sample_counts:
        subset = samples[:count]
        estimate = estimator_fn(subset)

        metrics["frobenius_norm"].append(frobenius_norm(estimate))
        metrics["frobenius_se"].append(frobenius_standard_error(subset))
        metrics["occupation_trace"].append(occupation_trace(estimate))
        metrics["spectral_radius"].append(spectral_radius(estimate))

        if previous is None:
            metrics["successive_diff"].append(float("nan"))
        else:
            metrics["successive_diff"].append(frobenius_norm(estimate - previous))
        previous = estimate

    return metrics


def collect_diagnostics(
    final_samples: np.ndarray,
    sample_counts: Iterable[int],
    *,
    include_mom: bool = True,
    mom_groups: int = None,
) -> Dict[str, Mapping[str, List[float]]]:
    """Compute diagnostics for requested estimators."""
    sample_counts = list(sample_counts)
    estimators: Dict[str, Callable[[np.ndarray], np.ndarray]] = {
        "Mean": mean_estimator,
    }
    if include_mom:
        estimators["Median-of-Means"] = lambda data: median_of_means_estimator(
            data, n_groups=mom_groups
        )

    diagnostics: Dict[str, Mapping[str, List[float]]] = {}
    for name, fn in estimators.items():
        diagnostics[name] = analyze_estimator(final_samples, sample_counts, fn)
    return diagnostics


def plot_diagnostics(
    sample_counts: Iterable[int],
    diagnostics: Mapping[str, Mapping[str, List[float]]],
    output_path: Path = None,
):
    """Plot convergence diagnostics and return the figure."""
    sample_counts = list(sample_counts)
    metric_defs: List[Tuple[str, str, str]] = [
        ("frobenius_norm", "Frobenius Norm of Estimate", "||⟨s⟩||₍fro₎"),
        ("frobenius_se", "Std Error of Frobenius Norm", "SE[||s||₍fro₎]"),
        ("successive_diff", "Successive Estimate Difference", "||Δ⟨s⟩||₍fro₎"),
        ("occupation_trace", "Trace of Estimate", "Tr(⟨s⟩)"),
        ("spectral_radius", "Spectral Radius (or σ_max)", "ρ(⟨s⟩)"),
    ]

    fig, axes = plt.subplots(3, 2, figsize=(12, 12))
    ax_list = axes.flatten()
    palette = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

    for idx, (metric_key, title, ylabel) in enumerate(metric_defs):
        ax = ax_list[idx]
        for color, (label, metrics) in zip(palette, diagnostics.items()):
            values = np.array(metrics[metric_key], dtype=float)
            if metric_key == "successive_diff":
                invalid = ~np.isfinite(values) | (values <= 0)
                values[invalid] = np.nan
                ax.set_yscale("log")
            ax.plot(sample_counts, values, marker="o", label=label, color=color)
        ax.set_title(title)
        ax.set_xlabel("Samples")
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", alpha=0.5)
        if idx == 0:
            ax.legend()

    ax_list[-1].axis("off")
    fig.suptitle("Entanglement Contour Convergence Diagnostics", fontsize=16)
    fig.tight_layout(rect=[0, 0.03, 1, 0.96])

    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300)

    return fig


In [ ]:
# Cell 2: Load data and run diagnostics
data_path = Path("cache/G_history_samples/N16x16/N16x16_C20_S500_nshNone_DW1_init-default_n_a0.5_markov_circuit_alpha_triv=30.npz")
final_samples = load_final_time_samples(data_path)

Nlayer = final_samples.shape[-1]
Nx, Ny = infer_Nx_Ny_from_filename(data_path, Nlayer)
y_cut = Ny // 2

contour_samples = compute_entanglement_contour_samples(
    final_samples,
    Nx,
    Ny,
    y_cut=y_cut,
    batch_size=8,
)
print(f"Contour samples shape: {contour_samples.shape}")


In [ ]:
sample_counts = [1, 10, 25, 50, 75, 100, 250, 500]

# Convergence diagnostics on entanglement contour samples
diagnostics = collect_diagnostics(
    contour_samples,
    sample_counts,
    include_mom=False,   # set False to drop the MoM curves
    mom_groups=5,       # optional override; None defaults to floor(sqrt(S))
)
diagnostics
# Cell 3: Plot (and optionally save)
fig = plot_diagnostics(
    sample_counts,
    diagnostics,
    output_path=None)
plt.show()
